In [ ]:
# Example breast cancer script, Tisch datasets can be read in a similar way to the Lung script
import scanpy as sc
import pandas as pd
import os

# ==========================================
# 1. Define your file paths
# ==========================================
# Update this to wherever you extracted the BC_counts folder
counts_dir = r"path_to\2102-Breastcancer_counts\export\BC_counts"
metadata_path = r"path_to\2103-Breastcancer_metadata.csv.gz"
output_file = r"path_to\Breast_PanCancer_Annotated.h5ad"

print("Starting processing...")

# ==========================================
# 2. Load the Count Matrix
# ==========================================
# Scanpy automatically looks for matrix.mtx, barcodes.tsv, and genes.tsv (or features.tsv) inside the folder.
try:
    print(f"Loading matrix from {counts_dir}...")
    # var_names='gene_symbols' tells Scanpy to use the gene names rather than Ensembl IDs
    adata = sc.read_10x_mtx(counts_dir, var_names='gene_symbols', cache=False)
except ValueError as e:
    print(f"Error reading 10x directory. Make sure matrix.mtx, barcodes.tsv, and genes.tsv exist in the folder.\nDetails: {e}")
    exit()

# Ensure gene names are unique (appends '-1', '-2' to duplicates to prevent Scanpy crashes)
adata.var_names_make_unique()

# ==========================================
# 3. Load and Format Metadata
# ==========================================
print(f"Loading metadata from {metadata_path}...")
# Pandas can natively read .csv.gz compressed files! 
# index_col=0 ensures the cell barcodes are set as the row index so they match the matrix.
metadata = pd.read_csv(metadata_path, compression='gzip', index_col=0)

# Sometimes metadata barcode formats differ slightly from the matrix (e.g., missing the '-1' suffix).
# Let's check if the barcodes match perfectly before merging.
matrix_barcodes = set(adata.obs_names)
meta_barcodes = set(metadata.index)

overlap = len(matrix_barcodes.intersection(meta_barcodes))
print(f"Matrix cells: {len(matrix_barcodes)} | Metadata cells: {len(meta_barcodes)} | Overlap: {overlap}")

if overlap == 0:
    print("WARNING: Zero overlap between matrix barcodes and metadata barcodes.")
    print("Example matrix barcode:", list(matrix_barcodes)[0])
    print("Example metadata barcode:", list(meta_barcodes)[0])
    # Common fix: SCope metadata often strips the "-1" from the end of 10x barcodes. 
    # If this happens, you can uncomment the line below to strip it from the matrix barcodes too:
    # adata.obs_names = adata.obs_names.str.replace("-1", "", regex=False)

# ==========================================
# 4. Attach Metadata and Save
# ==========================================
print("Attaching metadata to AnnData object...")
# This step automatically aligns the metadata to the correct cells based on the barcode index
adata.obs = metadata

print(f"Saving fully annotated dataset to {output_file}...")
adata.write_h5ad(output_file)
print("Success! Your .h5ad file is ready for analysis.")

: 